# Coral Vision — EfficientNet-B0 Training (Google Colab)

Trains the coral species classifier by transfer learning on EfficientNet-B0, evaluates it, and exports a `.tflite` file for the mobile app. Same logic as the local `training/` scripts in the project, just running here with a free GPU instead of your own computer.

## Before you run anything

1. **Turn on the GPU**: menu bar → `Runtime` → `Change runtime type` → Hardware accelerator → **GPU** → Save.
2. **Upload your dataset to Google Drive** as a single zip file:
   - Create a folder in your Drive called `CoralVision`.
   - Inside it, put a file named `dataset.zip`.
   - That zip should contain one folder per species (named exactly like the ids below), each full of that species' photos:
     ```
     dataset.zip
       bean_coral/
         img001.jpg
         img002.jpg
       brain_coral/
         ...
       bubble_coral/
         ...
     ```
3. Run the cells **top to bottom**, in order (▶ button on each cell, or `Runtime` → `Run all`).

**If your species list changes** (once PENRO confirms the real 5–10 species), update `SPECIES_IDS` in the config cell below — and update `src/config/species.js` in the app to match, in the same order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# Config — MUST match src/config/species.js (same species, same order)
# ---------------------------------------------------------------------------
import os

SPECIES_IDS = [
    "bean_coral",
    "brain_coral",
    "bubble_coral",
    "cauliflower_coral",
    "favia_coral",
    "lettuce_coral",
    "mushroom_coral",
    "crown_of_thorns_starfish",
]
NUM_CLASSES = len(SPECIES_IDS)

IMG_SIZE = (224, 224)          # EfficientNet-B0's native input size
BATCH_SIZE = 16
RANDOM_SEED = 42

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

STAGE1_EPOCHS = 15
STAGE1_LEARNING_RATE = 1e-3
STAGE2_EPOCHS = 20
STAGE2_LEARNING_RATE = 1e-5
STAGE2_UNFREEZE_FROM_LAYER = 200
EARLY_STOPPING_PATIENCE = 5

DRIVE_ROOT = "/content/drive/MyDrive/CoralVision"
DATASET_ZIP = f"{DRIVE_ROOT}/dataset.zip"
RAW_DIR = "/content/data/raw"                # unzipped raw images (in the Colab VM, not Drive — faster)
PROCESSED_DIR = "/content/data/processed"     # train/val/test split
MODELS_DIR = f"{DRIVE_ROOT}/models"           # saved to Drive so it survives when the session ends
EXPORT_DIR = f"{DRIVE_ROOT}/export"
REPORTS_DIR = f"{DRIVE_ROOT}/reports"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Config ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Unzip the dataset from Drive into the Colab VM's local disk
# ---------------------------------------------------------------------------
import zipfile

assert os.path.exists(DATASET_ZIP), (
    f"Dataset zip not found at {DATASET_ZIP} — upload dataset.zip to your "
    f"Google Drive under a 'CoralVision' folder first, then re-run this cell."
)

os.makedirs(RAW_DIR, exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, "r") as z:
    z.extractall(RAW_DIR)

print("Per-species image counts:")
for species in SPECIES_IDS:
    folder = os.path.join(RAW_DIR, species)
    count = len(os.listdir(folder)) if os.path.isdir(folder) else 0
    flag = "" if count >= 20 else "  <-- consider collecting more"
    print(f"  {species}: {count if count else 'MISSING FOLDER'}{flag}")

In [ ]:
# ---------------------------------------------------------------------------
# Split into train / val / test (70/15/15, stratified per species)
# ---------------------------------------------------------------------------
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".heic", ".webp"}

def find_images(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

if os.path.exists(PROCESSED_DIR):
    shutil.rmtree(PROCESSED_DIR)

for species in SPECIES_IDS:
    species_dir = Path(RAW_DIR) / species
    if not species_dir.is_dir():
        continue
    images = find_images(species_dir)
    if len(images) < 10:
        print(f"WARNING: '{species}' has only {len(images)} image(s) — may not be enough to train well.")
    if not images:
        continue

    train_val, test = train_test_split(images, test_size=TEST_SPLIT, random_state=RANDOM_SEED)
    relative_val_size = VAL_SPLIT / (TRAIN_SPLIT + VAL_SPLIT)
    train, val = train_test_split(train_val, test_size=relative_val_size, random_state=RANDOM_SEED)

    for split_name, split_images in (("train", train), ("val", val), ("test", test)):
        dest_dir = Path(PROCESSED_DIR) / split_name / species
        dest_dir.mkdir(parents=True, exist_ok=True)
        for src in split_images:
            shutil.copy2(src, dest_dir / src.name)

    print(f"{species}: {len(train)} train / {len(val)} val / {len(test)} test")

In [ ]:
# ---------------------------------------------------------------------------
# Build train/val datasets, with augmentation applied only to training data
# ---------------------------------------------------------------------------
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input

print("GPU available:", tf.config.list_physical_devices("GPU"))
tf.random.set_seed(RANDOM_SEED)

train_dir = os.path.join(PROCESSED_DIR, "train")
val_dir = os.path.join(PROCESSED_DIR, "val")

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, labels="inferred", label_mode="categorical",
    class_names=SPECIES_IDS, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    seed=RANDOM_SEED, shuffle=True,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, labels="inferred", label_mode="categorical",
    class_names=SPECIES_IDS, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    shuffle=False,
)

augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name="augmentation")

def prep_train(image, label):
    image = augmentation(image, training=True)
    return preprocess_input(image), label

def prep_eval(image, label):
    return preprocess_input(image), label

train_ds = train_ds.map(prep_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(prep_eval, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# ---------------------------------------------------------------------------
# Build the model: EfficientNet-B0 base (frozen) + new classification head
# ---------------------------------------------------------------------------
def build_model():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3), pooling="avg",
    )
    base.trainable = False  # Stage 1: frozen
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    return models.Model(inputs, outputs, name="coral_vision_efficientnet_b0")

model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE1_LEARNING_RATE),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
model.summary()

In [ ]:
# ---------------------------------------------------------------------------
# Stage 1: train only the new classification head (base frozen)
# ---------------------------------------------------------------------------
checkpoint_path = os.path.join(MODELS_DIR, "best_model.keras")
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
]

print("=== Stage 1: training classification head (base frozen) ===")
history1 = model.fit(train_ds, validation_data=val_ds, epochs=STAGE1_EPOCHS, callbacks=callbacks)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: unfreeze the top of the base and fine-tune at a low learning rate
# ---------------------------------------------------------------------------
print("=== Stage 2: fine-tuning top of EfficientNet-B0 base ===")
base = model.layers[1]
base.trainable = True
for layer in base.layers[:STAGE2_UNFREEZE_FROM_LAYER]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE2_LEARNING_RATE),
    loss="categorical_crossentropy", metrics=["accuracy"],
)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=STAGE2_EPOCHS, callbacks=callbacks)

final_path = os.path.join(MODELS_DIR, "coral_vision_final.keras")
model.save(final_path)

import json
with open(os.path.join(MODELS_DIR, "class_names.json"), "w") as f:
    json.dump(SPECIES_IDS, f, indent=2)

print(f"\nSaved best checkpoint: {checkpoint_path}")
print(f"Saved final model: {final_path}")
print("(Both are in your Google Drive under CoralVision/models/, so they'll still be there next time you open this notebook.)")

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate on the held-out test set — this is the thesis accuracy evaluation
# ---------------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

test_dir = os.path.join(PROCESSED_DIR, "test")
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="categorical",
    class_names=SPECIES_IDS, image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
)
y_true = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds], axis=0)
test_ds_eval = test_ds.map(lambda x, y: (preprocess_input(x), y))

probs = model.predict(test_ds_eval)
y_pred = np.argmax(probs, axis=1)

overall_accuracy = float(np.mean(y_true == y_pred))
print(f"Overall test accuracy: {overall_accuracy:.4f}\n")

report_text = classification_report(y_true, y_pred, target_names=SPECIES_IDS, digits=4)
print(report_text)

with open(os.path.join(REPORTS_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Overall test accuracy: {overall_accuracy:.4f}\n\n{report_text}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(SPECIES_IDS))); ax.set_yticks(range(len(SPECIES_IDS)))
ax.set_xticklabels(SPECIES_IDS, rotation=45, ha="right"); ax.set_yticklabels(SPECIES_IDS)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (accuracy: {overall_accuracy:.2%})")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(REPORTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

print(f"\nSaved report + confusion matrix to your Drive under CoralVision/reports/")

In [ ]:
# ---------------------------------------------------------------------------
# Export to TensorFlow Lite — the format the mobile app actually loads
# ---------------------------------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

tflite_path = os.path.join(EXPORT_DIR, "coral_vision_efficientnet_b0.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

labels_path = os.path.join(EXPORT_DIR, "labels.txt")
with open(labels_path, "w") as f:
    f.write("\n".join(SPECIES_IDS) + "\n")

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"Wrote {tflite_path} ({size_mb:.1f} MB)")
print(f"Wrote {labels_path}")
print("\nBoth files are saved in your Google Drive under CoralVision/export/.")
print("Next: download them from Drive (or send them to me) and I'll wire them into")
print("the mobile app's inference.js in place of the mock predictor.")